# Notebook Overview

**Purpose:**
This notebook benchmarks Optuna-selected LSTM and xLSTM models on the AAPL test set across multiple forecast horizons and feature configurations. For each combination of model type (LSTM, xLSTM), feature set (price-only vs. price plus sentiment), and horizon, it loads the corresponding Optuna study, selects a representative Pareto-optimal trial, restores the saved checkpoint, and evaluates predictive performance on the denormalised price level.

**Inputs:**
- Sentiment-augmented time series CSV: `../data/ts_with_sentiment/AAPL_ts_with_sentiment.csv`
- Optuna studies (SQLite): `../data/optuna_studies/BachelorThesis.db`
- Trial artifacts stored in Optuna metadata:
  - Model configuration (`trial.user_attrs["config"]`)
  - Checkpoint path (`trial.user_attrs["best_checkpoint_path"]`)
- Evaluation parameters defined in the configuration block:
  - Train/validation split ratios, window size, horizons, stride
  - Feature sets and normalisation method (must match training)

**Outputs:**
- Console logs per configuration including checkpoint selection and test-set metrics.
- Test-set plots (true vs. predicted) in the original price scale, typically shown for the final horizon step (t+H) when `stride = 1`.
- A consolidated comparison table (`df_display`) summarising denormalised test RMSE and R² across all evaluated configurations, optionally grouped by model type.

**Process Summary:**
For each study identified by `(model_type, horizon, n_features)`, the notebook loads the Optuna study from the SQLite storage and selects a Pareto-optimal trial using a deterministic rule (minimum RMSE with R² as a tie-breaker). It reconstructs the test set using the same preprocessing and normalisation pipeline as training, restores the selected checkpoint, generates multi-horizon predictions, denormalises predictions and targets back to the original price level, and computes RMSE and R² on the test set. Results are visualised via per-configuration prediction plots and aggregated into a structured table for direct comparison between architectures and feature sets.


In [ ]:
# =========================
# Imports
# =========================
import os
from pathlib import Path
from typing import Dict, Any, List, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from sklearn.metrics import r2_score, root_mean_squared_error

import torch

from src.model_wrapper import Model, set_global_seed
from src.data_prep import PrepAndDataLoader

# Optuna Logging reduzieren
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Info] Using device: {DEVICE}")


In [ ]:
# =========================
# Global configuration
# =========================

# --- Data settings ---
DATA_FILE: str = "../data/ts_with_sentiment/AAPL_ts_with_sentiment.csv"
TARGET_COL: str = "Close"
TRAIN_SPLIT: float = 0.6
VAL_SPLIT: float = 0.2

# Feature sets: 1 Feature vs. 5 Features
FEATURE_SETS: Dict[str, List[str]] = {
    "price_only": ["Close"],  # 1 Feature
    "price_plus_sentiment": [  # 5 Features
        "Close",
        "pos_mean",
        "neut_mean",
        "neg_mean",
        "news_count",
    ],
}

# --- Window / horizon settings ---
WINDOW_SIZE: int = 60
HORIZONS: List[int] = [1, 3, 5, 10, 15]
STRIDE: int = 1

# --- Normalization settings (muss zum Training passen) ---
NORMALISE: bool = True
NORM_METHOD: Literal["percentage", "minmax"] = "percentage"

# --- Model / Optuna settings ---
MODEL_TYPES: List[str] = ["xLSTM", "LSTM"]

DB_PATH = Path("../data/optuna_studies/BachelorThesis.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
STORAGE_URL = f"sqlite:///{DB_PATH.as_posix()}"

CHECKPOINT_DIR_TEMPLATE: str = (
    "../data/optuna_checkpoints/{model_type}_optuna_H{horizon}_F{n_features}"
)

# --- Batchgröße für Predictions ---
BATCH_SIZE: int = 128

# --- Seed ---
GLOBAL_SEED: int = 42

# Asset-Name für Plot-Titel (aus Dateinamen)
asset_name = Path(DATA_FILE).stem.split("_")[0]  # z.B. "AAPL"

In [ ]:
# =========================
# Helper: besten Trial aus Study wählen
# =========================

def select_best_trial(study: optuna.Study) -> optuna.trial.FrozenTrial:
    """
    Wählt einen 'besten' Trial aus einer (multi-objective) Optuna-Study.

    Kriterium (intern, auf Basis der Optuna-Objektivwerte):
      1) minimaler RMSE (values[0])
      2) bei Gleichstand maximaler R² (values[1])

    Diese Werte werden nur zur Wahl des Trials verwendet und
    später nicht in Plots/Tabellen angezeigt.
    """
    if hasattr(study, "best_trials") and study.best_trials:
        pareto_trials = study.best_trials
    else:
        pareto_trials = [study.best_trial]

    best = min(pareto_trials, key=lambda t: (t.values[0], -t.values[1]))
    return best


# =========================
# Helper: Datenvorbereitung (TEST-Set)
# =========================

def prepare_test_data_for_config(
        feature_cols: List[str],
        horizon: int,
) -> Dict[str, Any]:
    """
    Erzeugt für gegebene Feature-Spalten und Horizont die TEST-Daten
    im gleichen Format wie beim Training.

    Rückgabe:
        {
            "prep": PrepAndDataLoader,
            "X_te": np.ndarray [N_te, W, F],
            "y_te": np.ndarray [N_te, H, 1],
            "dates_te": np.ndarray [N_te, H],
            "baseT_te": np.ndarray [N_te, 1] oder None,
        }
    """
    set_global_seed(GLOBAL_SEED)

    prep = PrepAndDataLoader(
        filename=DATA_FILE,
        training_split=TRAIN_SPLIT,
        validation_split=VAL_SPLIT,
        cols=feature_cols,
        target_col=TARGET_COL,
    )

    tmp_model = Model()
    X_tr, y_tr, X_va, y_va, X_te, y_te = tmp_model.prepare_data_from_prep(
        prep,
        normalise=NORMALISE,
        window_size=WINDOW_SIZE,
        prediction_range=horizon,
        norm_method=NORM_METHOD,
        stride=STRIDE,
        verbose=0,
    )

    dates_te = prep.get_prediction_dates(
        "test",
        window_size=WINDOW_SIZE,
        prediction_range=horizon,
        stride=STRIDE,
    )  # [N_te, H]

    baseT_te = tmp_model._cache.get("baseT_te", None)

    return {
        "prep": prep,
        "X_te": X_te,
        "y_te": y_te,
        "dates_te": dates_te,
        "baseT_te": baseT_te,
    }


# =========================
# Helper: Denormalisation + Test-Metriken
# =========================

def denormalise_test_and_metrics(
        prep: PrepAndDataLoader,
        y_test_scaled: np.ndarray,  # [N, H, 1]
        y_pred_scaled: np.ndarray,  # [N, H]
        baseT_te: np.ndarray | None,
) -> Dict[str, Any]:
    """
    Denormalisiert Vorhersagen und Targets (sofern NORMALISE=True)
    und berechnet RMSE/R² im Original-Level auf dem TEST-Set.
    """
    y_pred_scaled_3d = y_pred_scaled[..., None]  # [N, H, 1]
    y_true_scaled_3d = y_test_scaled  # [N, H, 1]

    if NORMALISE:
        if NORM_METHOD == "percentage":
            if baseT_te is None:
                raise ValueError(
                    "baseT_te ist None, wird aber für percentage-Normalisierung benötigt."
                )

            y_pred_level = prep.denormalise(
                y_pred_scaled_3d,
                method="percentage",
                base_values=baseT_te,
                normalise=True,
            ).squeeze(-1)  # [N, H]

            y_true_level = prep.denormalise(
                y_true_scaled_3d,
                method="percentage",
                base_values=baseT_te,
                normalise=True,
            ).squeeze(-1)

        elif NORM_METHOD == "minmax":
            y_pred_level = prep.denormalise(
                y_pred_scaled_3d,
                method="minmax",
                base_values=None,
                normalise=True,
            ).squeeze(-1)

            y_true_level = prep.denormalise(
                y_true_scaled_3d,
                method="minmax",
                base_values=None,
                normalise=True,
            ).squeeze(-1)
        else:
            raise ValueError(f"Unknown normalization method: {NORM_METHOD}")
    else:
        y_pred_level = y_pred_scaled
        y_true_level = y_test_scaled.squeeze(-1)

    # Flatten für Metriken
    y_true_flat = y_true_level.reshape(-1)
    y_pred_flat = y_pred_level.reshape(-1)

    test_rmse_level = float(root_mean_squared_error(y_true_flat, y_pred_flat))
    test_r2_level = float(r2_score(y_true_flat, y_pred_flat))

    return {
        "y_true_level": y_true_level,
        "y_pred_level": y_pred_level,
        "test_rmse_level": test_rmse_level,
        "test_r2_level": test_r2_level,
    }


# =========================
# Helper: Plot-Funktion (TEST, denormalisiert)
# =========================

def plot_test_predictions(
        dates_te: np.ndarray,  # [N, H]
        y_true_level: np.ndarray,  # [N, H]
        y_pred_level: np.ndarray,  # [N, H]
        model_type: str,
        horizon: int,
        n_features: int,
        feature_set_name: str,
        test_rmse_level: float,
        test_r2_level: float,
):
    """
    Plot True vs. Pred für den letzten Horizont t+H auf den TEST-Daten
    in der denormalisierten Originalskala.
    """
    if STRIDE == 1:
        h_idx = horizon - 1  # letzter Horizont t+H

        dates_h = dates_te[:, h_idx]
        true_h = y_true_level[:, h_idx]
        pred_h = y_pred_level[:, h_idx]

        plt.figure(figsize=(12, 5))

        plt.plot(dates_h, true_h, label=f"True (t+{horizon})")
        plt.plot(dates_h, pred_h, label=f"Pred (t+{horizon})")

        title_main = (
            f"{asset_name} – {model_type} | Horizon H={horizon} | "
            f"Features={n_features} ({feature_set_name}) – TEST"
        )
        title_metrics = f"Test RMSE={test_rmse_level:.4f}, Test R²={test_r2_level:.3f}"

        plt.title(f"{title_main}\n{title_metrics}")
        plt.xlabel("Date (Test set)")
        plt.ylabel(TARGET_COL)
        plt.legend()
        plt.grid(True)
        plt.gcf().autofmt_xdate()
        plt.tight_layout()
        plt.show()
    else:
        # Allgemeinerer Fall: alle Horizonte mergen (falls STRIDE>1)
        dates_flat = dates_te.reshape(-1)
        true_flat = y_true_level.reshape(-1)
        pred_flat = y_pred_level.reshape(-1)

        df_plot = pd.DataFrame(
            {
                "date": dates_flat,
                "true": true_flat,
                "pred": pred_flat,
            }
        )

        df_plot = df_plot.groupby("date", as_index=True).mean().sort_index()

        plt.figure(figsize=(12, 5))
        plt.plot(df_plot.index, df_plot["true"], label="True (merged horizons)")
        plt.plot(df_plot.index, df_plot["pred"], label="Pred (merged horizons)")

        title_main = (
            f"{asset_name} – {model_type} | Horizon H={horizon} | "
            f"Features={n_features} ({feature_set_name}), stride={STRIDE} – TEST"
        )
        title_metrics = f"Test RMSE={test_rmse_level:.4f}, Test R²={test_r2_level:.3f}"

        plt.title(f"{title_main}\n{title_metrics}")
        plt.xlabel("Date (Test set)")
        plt.ylabel(TARGET_COL)
        plt.legend()
        plt.grid(True)
        plt.gcf().autofmt_xdate()
        plt.tight_layout()
        plt.show()

In [ ]:
# =========================
# Hauptschleife:
# =========================

results: List[Dict[str, Any]] = []

for model_type in sorted(MODEL_TYPES):
    for feature_set_name, feature_cols in FEATURE_SETS.items():
        n_features = len(feature_cols)

        for horizon in sorted(HORIZONS):
            study_name = f"{model_type}_H{horizon}_F{n_features}"
            ckpt_dir = CHECKPOINT_DIR_TEMPLATE.format(
                model_type=model_type,
                horizon=horizon,
                n_features=n_features,
            )

            print(f"\n[INFO] Processing Study: {study_name}")

            # --- Study laden ---
            try:
                study = optuna.load_study(
                    study_name=study_name,
                    storage=STORAGE_URL,
                )
            except Exception as e:
                print(f"[WARN] Could not load study '{study_name}': {e}")
                continue

            # --- besten Trial bestimmen ---
            try:
                best_trial = select_best_trial(study)
            except Exception as e:
                print(f"[WARN] Could not select best trial for '{study_name}': {e}")
                continue

            # User-Attribute
            best_ckpt = best_trial.user_attrs.get("best_checkpoint_path", None)
            best_config = best_trial.user_attrs.get("config", None)

            if best_ckpt is None or best_config is None:
                print(
                    f"[WARN] Missing checkpoint or config in best trial of '{study_name}'."
                )
                continue

            if not os.path.isfile(best_ckpt):
                print(f"[WARN] Checkpoint file not found: {best_ckpt}")
                continue

            print(f"  -> Using checkpoint: {best_ckpt}")

            # --- TEST-Daten für diese Konfiguration vorbereiten ---
            data_dict = prepare_test_data_for_config(
                feature_cols=feature_cols,
                horizon=horizon,
            )

            prep = data_dict["prep"]
            X_te = data_dict["X_te"]
            y_te = data_dict["y_te"]
            dates_te = data_dict["dates_te"]
            baseT_te = data_dict["baseT_te"]

            # --- Modell neu aufbauen und Checkpoint laden ---
            eval_model = Model()
            eval_model.build(config=best_config, model_type=model_type)
            eval_model.load(best_ckpt)

            # --- Vorhersagen auf TEST-Daten (Skalenraum) ---
            y_test_scaled = y_te  # [N, H, 1]
            y_pred_scaled = eval_model.predict_multi_horizon(
                X_te,
                batch_size=BATCH_SIZE,
                verbose=0,
            )  # [N, H]

            # --- Denormalisieren & Test-Metriken im Original-Level ---
            metrics_dict = denormalise_test_and_metrics(
                prep=prep,
                y_test_scaled=y_test_scaled,
                y_pred_scaled=y_pred_scaled,
                baseT_te=baseT_te,
            )

            y_true_level = metrics_dict["y_true_level"]
            y_pred_level = metrics_dict["y_pred_level"]
            test_rmse_level = metrics_dict["test_rmse_level"]
            test_r2_level = metrics_dict["test_r2_level"]

            print(
                f"  -> TEST (LEVEL) RMSE={test_rmse_level:.6f}, R²={test_r2_level:.4f}"
            )

            # --- Plot: True vs. Pred (TEST, Original-Level) ---
            plot_test_predictions(
                dates_te=dates_te,
                y_true_level=y_true_level,
                y_pred_level=y_pred_level,
                model_type=model_type,
                horizon=horizon,
                n_features=n_features,
                feature_set_name=feature_set_name,
                test_rmse_level=test_rmse_level,
                test_r2_level=test_r2_level,
            )

            # --- Ergebnisse für Vergleichstabelle sammeln ---
            results.append(
                {
                    "asset": asset_name,
                    "model_type": model_type,
                    "feature_set": feature_set_name,
                    "n_features": n_features,
                    "horizon": horizon,
                    "test_rmse_level": test_rmse_level,
                    "test_r2_level": test_r2_level,
                    "checkpoint": best_ckpt,
                    "study_name": study_name,
                }
            )



In [ ]:
# =========================
# Vergleichstabelle (nur TEST, denormalisiert)
# =========================

if len(results) == 0:
    print("[WARN] No results collected. Please check paths and study names.")
else:
    df_results = pd.DataFrame(results)

    # Sortierung: Modelltyp, Feature-Anzahl, Horizont
    df_results = df_results.sort_values(
        by=["model_type", "n_features", "horizon"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    display_cols = [
        "asset",
        "model_type",
        "feature_set",
        "n_features",
        "horizon",
        "test_rmse_level",
        "test_r2_level",
    ]

    df_display = df_results[display_cols].copy()
    df_display["test_rmse_level"] = df_display["test_rmse_level"].round(4)
    df_display["test_r2_level"] = df_display["test_r2_level"].round(4)

    print("\n=========================")
    print("Vergleich xLSTM vs. LSTM | 1 Feature vs. 5 Features")
    print("Basis: TEST-Predictions, denormalisierte Werte")
    print("=========================\n")

    display(df_display)

    # Optional: nach Modelltyp gruppiert anzeigen
    for model_type in df_display["model_type"].unique():
        print(f"\n--- {model_type} ---")
        display(df_display[df_display["model_type"] == model_type])